In [6]:
#!/usr/bin/env python3
# run_hs2swe_save_station_nc_season_Nov1_Apr30_STANDALONE.py
#
# Standalone: includes HS2SWE() + batch runner.
# Reads ALL station CSVs in INDIR, splits into seasons (Nov 1 -> Apr 30),
# runs HS2SWE, and writes ONE NetCDF per station with ALL kept seasons inside.
#
# NetCDF structure (dim order):
#   season, dos (Day Of Season), layer
#
# Variables stored as:
#   HS_layer_cm, RHO, OVB, AGE : (season, dos, layer)
#   DIA                        : (season, dos, diag)
#   SWE_mm, HS_obs_cm          : (season, dos)
#
# Output dir:
# /Users/jakobwerkgarner/code/mt_dsnow/model_diff/R_comparisson/output/HS_SWE_by_station/HS_SWE_by_station_hs2swe_only

from __future__ import annotations

import re
import time
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from netCDF4 import Dataset

# -----------------------------
# USER PATHS
# -----------------------------
INDIR = Path("/Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_data/output/HS_SWE_by_station/")
OUTDIR = Path("/Users/jakobwerkgarner/code/mt_dsnow/model_diff/R_comparisson/output/HS_SWE_by_station/HS_SWE_by_station_hs2swe_only")
OUTDIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# SETTINGS
# -----------------------------
MIN_DAYS_PER_SEASON = 10
MIN_LAYERS_TO_SAVE = 1  # set to 4 to keep only seasons with >3 active layers
TIMEKEEPING = 1         # keep your printouts


# -----------------------------
# HS2SWE (standalone)
# -----------------------------
def HS2SWE(
    idata,
    RhoNew=113.7, RhoMax=571.6, SnoTemp=-0.000, Visc=6.051e7, DQMultInc=0.1, DQMultMax=5, HsAcc=2,
    c1=2.8e-6, c2=0.042, c3=0.046, c4=0.081, c5=0.018, g=9.81, dt=86400,
    return_MR=False
):
    PAR = {"c1": c1, "c2": c2, "c3": c3, "c4": c4, "c5": c5, "g": g, "dt": dt}

    idata = np.asarray(idata, dtype=float)
    assert idata.ndim == 2, "idata must be 2D: (time, series)"
    assert np.all(idata[~np.isnan(idata)] >= 0), "Negative values in input data"

    odata = np.zeros_like(idata)

    MR_out = None
    if return_MR:
        MR_out = []

    if TIMEKEEPING:
        np.set_printoptions(threshold=np.inf)
        start = time.time()

    for serix in range(idata.shape[1]):
        print(f"Running station series: {serix + 1}")
        if TIMEKEEPING:
            start_index = time.time()

        psdix = np.where((~np.isnan(idata[:, serix])) & (idata[:, serix] > 0))[0]
        if psdix.size == 0:
            if return_MR:
                MR_out.append({
                    "HS":  np.zeros((1, idata.shape[0]), dtype=float),
                    "RHO": np.full((1, idata.shape[0]), np.nan, dtype=float),
                    "OVB": np.full((1, idata.shape[0]), np.nan, dtype=float),
                    "AGE": np.full((1, idata.shape[0]), np.nan, dtype=float),
                    "DIA": np.zeros((5, idata.shape[0]), dtype=float),
                })
            continue

        if return_MR:
            MR_station = {
                "HS":  np.zeros((1, idata.shape[0]), dtype=float),
                "RHO": np.full((1, idata.shape[0]), np.nan, dtype=float),
                "OVB": np.full((1, idata.shape[0]), np.nan, dtype=float),
                "AGE": np.full((1, idata.shape[0]), np.nan, dtype=float),
                "DIA": np.zeros((5, idata.shape[0]), dtype=float),
            }

        sepix = np.concatenate(([0], np.where(np.diff(psdix) > 1)[0] + 1, [len(psdix)]))

        for tsrix in range(len(sepix) - 1):
            i0 = psdix[sepix[tsrix]]
            i1 = psdix[sepix[tsrix + 1] - 1]
            seg_idx = np.arange(i0, i1 + 1, dtype=int)

            HS = np.concatenate(([0], idata[seg_idx, serix]))

            MR = {
                "HS":  np.zeros((1, len(HS))),
                "RHO": np.ones((1, len(HS))) * RhoMax,
                "OVB": np.zeros((1, len(HS))),
                "AGE": np.arange(len(HS)).reshape(1, -1),
                "DIA": np.zeros((5, len(HS))),
            }

            for tn in range(1, len(HS)):
                MR["RHO"][:, tn] = np.minimum(
                    MR["RHO"][:, tn - 1] + MR["RHO"][:, tn - 1] * PAR["dt"] * (
                        MR["OVB"][:, tn - 1] * PAR["g"] / (Visc * np.exp(PAR["c4"] * SnoTemp + PAR["c5"] * MR["RHO"][:, tn - 1])) +
                        PAR["c1"] * np.exp(-PAR["c2"] * SnoTemp - PAR["c3"] * np.maximum(0, MR["RHO"][:, tn - 1] - RhoNew))
                    ),
                    RhoMax
                )

                MR["HS"][:, tn] = MR["HS"][:, tn - 1] / (MR["RHO"][:, tn] / MR["RHO"][:, tn - 1])

                HSmod = np.sum(MR["HS"][:, tn])
                HSmeas = HS[tn]

                if HSmeas > HSmod and tn == 1:
                    nlix = MR["HS"].shape[0] + 1
                    MR["HS"]  = np.vstack([MR["HS"],  np.zeros((1, len(HS)))])
                    MR["HS"][nlix - 1, tn] = HSmeas - np.sum(MR["HS"][:, tn])
                    MR["RHO"] = np.vstack([MR["RHO"], np.ones((1, len(HS))) * RhoNew])
                    MR["OVB"] = np.vstack([MR["OVB"], np.zeros((1, len(HS)))])
                    MR["AGE"] = np.vstack([MR["AGE"], np.zeros((1, len(HS)))])
                    MR["AGE"][nlix - 1, tn:] = np.arange(len(HS) - tn)

                    if HSmeas < HS[tn - 1]:
                        MR["DIA"][0, tn] = 1

                elif HSmeas > HSmod + HsAcc:
                    nlix = MR["HS"].shape[0] + 1
                    MR["HS"]  = np.vstack([MR["HS"],  np.zeros((1, len(HS)))])
                    MR["HS"][nlix - 1, tn] = HSmeas - np.sum(MR["HS"][:, tn])
                    MR["RHO"] = np.vstack([MR["RHO"], np.ones((1, len(HS))) * RhoNew])
                    MR["OVB"] = np.vstack([MR["OVB"], np.zeros((1, len(HS)))])
                    MR["AGE"] = np.vstack([MR["AGE"], np.zeros((1, len(HS)))])
                    MR["AGE"][nlix - 1, tn:] = np.arange(len(HS) - tn)

                    if HSmeas < HS[tn - 1]:
                        MR["DIA"][0, tn] = 1

                elif HSmeas == HSmod:
                    MR["DIA"][1, tn] = 0

                elif HSmeas > HSmod:
                    MR["DIA"][1, tn] = HSmeas - HSmod
                    DQMultCur = 1
                    while np.mean(MR["RHO"][:, tn]) < RhoMax and HSmeas > np.sum(MR["HS"][:, tn]) and DQMultCur < DQMultMax:
                        DQMultCur += DQMultInc
                        MR["RHO"][:, tn] = np.minimum(
                            MR["RHO"][:, tn - 1] + MR["RHO"][:, tn - 1] * PAR["dt"] * (
                                MR["OVB"][:, tn - 1] * PAR["g"] / (Visc * np.exp(PAR["c4"] * SnoTemp + PAR["c5"] * MR["RHO"][:, tn - 1])) +
                                PAR["c1"] * np.exp(-PAR["c2"] * SnoTemp - PAR["c3"] * np.maximum(0, MR["RHO"][:, tn - 1] - RhoNew))
                            ) / DQMultCur,
                            RhoMax
                        )
                        MR["HS"][:, tn] = MR["HS"][:, tn - 1] / (MR["RHO"][:, tn] / MR["RHO"][:, tn - 1])

                    MR["DIA"][2, tn] = HSmeas - np.sum(MR["HS"][:, tn])
                    MR["DIA"][3, tn] = -DQMultCur
                    if HSmeas > np.sum(MR["HS"][:, tn]):
                        MR["DIA"][4, tn] = -1

                elif HSmeas < HSmod:
                    MR["DIA"][1, tn] = HSmeas - HSmod
                    DQMultCur = 1
                    while np.mean(MR["RHO"][:, tn]) < RhoMax and HSmeas < np.sum(MR["HS"][:, tn]) and DQMultCur < DQMultMax:
                        DQMultCur += DQMultInc
                        MR["RHO"][:, tn] = np.minimum(
                            (MR["RHO"][:, tn - 1] + MR["RHO"][:, tn - 1] * PAR["dt"] * (
                                MR["OVB"][:, tn - 1] * PAR["g"] / (Visc * np.exp(PAR["c4"] * SnoTemp + PAR["c5"] * MR["RHO"][:, tn - 1])) +
                                PAR["c1"] * np.exp(-PAR["c2"] * SnoTemp - PAR["c3"] * np.maximum(0, MR["RHO"][:, tn - 1] - RhoNew))
                            ) * DQMultCur),
                            RhoMax
                        )
                        MR["HS"][:, tn] = MR["HS"][:, tn - 1] / (MR["RHO"][:, tn] / MR["RHO"][:, tn - 1])

                    MR["DIA"][2, tn] = HSmeas - np.sum(MR["HS"][:, tn])
                    MR["DIA"][3, tn] = DQMultCur

                    if HSmeas < np.sum(MR["HS"][:, tn]):
                        for lix in range(MR["HS"].shape[0] - 1, -1, -1):
                            MR["HS"][lix, tn] = HSmeas - np.sum(MR["HS"][:lix, tn])
                            if MR["HS"][lix, tn] >= 0:
                                break
                            else:
                                MR["HS"][lix, tn] = 0
                        MR["DIA"][4, tn] = 1
                else:
                    raise Exception("Unexpected case")

                nlix = MR["HS"].shape[0]
                MR["OVB"][nlix - 1, tn] = 0
                for lix in range(MR["HS"].shape[0] - 2, -1, -1):
                    MR["OVB"][lix, tn] = np.sum(MR["HS"][lix + 1:, tn] * MR["RHO"][lix + 1:, tn] / 100)

            SWE = np.sum(MR["HS"] * MR["RHO"] / 100, axis=0)  # mm
            odata[seg_idx, serix] = SWE[1:]

            if return_MR:
                Tfull = idata.shape[0]
                Lseg = MR["HS"].shape[0]

                if MR_station["HS"].shape[0] < Lseg:
                    addL = Lseg - MR_station["HS"].shape[0]
                    MR_station["HS"]  = np.vstack([MR_station["HS"],  np.zeros((addL, Tfull))])
                    MR_station["RHO"] = np.vstack([MR_station["RHO"], np.full((addL, Tfull), np.nan)])
                    MR_station["OVB"] = np.vstack([MR_station["OVB"], np.full((addL, Tfull), np.nan)])
                    MR_station["AGE"] = np.vstack([MR_station["AGE"], np.full((addL, Tfull), np.nan)])

                MR_station["HS"][:Lseg, seg_idx]  = MR["HS"][:, 1:]
                MR_station["RHO"][:Lseg, seg_idx] = MR["RHO"][:, 1:]
                MR_station["OVB"][:Lseg, seg_idx] = MR["OVB"][:, 1:]
                MR_station["AGE"][:Lseg, seg_idx] = MR["AGE"][:, 1:]
                MR_station["DIA"][:, seg_idx] += MR["DIA"][:, 1:]

        if TIMEKEEPING:
            end_index = time.time()
            print(f"Elapsed time for series {serix + 1}: {end_index - start_index} s")

        if return_MR:
            MR_out.append(MR_station)

    odata[idata == 0] = 0

    if TIMEKEEPING:
        end = time.time()
        print(f"Average time per series: {(end - start)/idata.shape[1]} s")

    if return_MR:
        return odata, MR_out
    return odata


# -----------------------------
# Season logic (Nov 1 -> Apr 30)
# -----------------------------
def season_label(d: pd.Timestamp) -> int:
    return int(d.year + 1) if d.month >= 11 else int(d.year)

def in_season_window(d: pd.Timestamp, season: int) -> bool:
    start = pd.Timestamp(year=season - 1, month=11, day=1)
    end = pd.Timestamp(year=season, month=4, day=30)
    return (d >= start) and (d <= end)


# -----------------------------
# DOS helpers (Nov 1 -> Apr 30)
# -----------------------------
def season_start_end(season: int) -> Tuple[pd.Timestamp, pd.Timestamp]:
    start = pd.Timestamp(year=season - 1, month=11, day=1)
    end = pd.Timestamp(year=season, month=4, day=30)
    return start, end

def dos_length(season: int) -> int:
    start, end = season_start_end(season)
    return int((end - start).days) + 1

def dos_index(dates: pd.Series, season: int) -> np.ndarray:
    start, _ = season_start_end(season)
    return ((dates.dt.normalize() - start).dt.days + 1).to_numpy(dtype=int)


# -----------------------------
# Helpers
# -----------------------------
def station_name_from_file(p: Path) -> str:
    return re.sub(r"_hs_swe_obs.*$", "", p.stem)

def ensure_first_zero(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float).copy()
    if x.size > 0 and (not np.isfinite(x[0]) or x[0] != 0.0):
        x[0] = 0.0
    return x

def choose_hs_column(df: pd.DataFrame) -> np.ndarray:
    for c in ["hs", "HS_meas_m", "HS_meas", "HS"]:
        if c in df.columns:
            return pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
    raise ValueError("No HS column found (tried: hs, HS_meas_m, HS_meas, HS)")

def max_active_layers(hs_layers: np.ndarray) -> int:
    M = np.asarray(hs_layers, dtype=float)
    M = np.nan_to_num(M, nan=0.0)
    return int(np.sum(np.any(M > 0, axis=1)))


# -----------------------------
# NetCDF writer
# dims: season, dos, layer; diag dimension for DIA
# -----------------------------
def write_station_hs2swe_nc(
    out_nc: Path,
    seasons: List[int],
    ndos: int,
    HS_cm: np.ndarray,     # (S,D,L)
    RHO: np.ndarray,       # (S,D,L)
    OVB: np.ndarray,       # (S,D,L)
    AGE: np.ndarray,       # (S,D,L)
    DIA: np.ndarray,       # (S,D,5)
    SWE_mm: np.ndarray,    # (S,D)
    HS_obs_cm: np.ndarray  # (S,D)
) -> None:
    S, D, L = HS_cm.shape
    if D != ndos:
        raise ValueError(f"ndos mismatch: ndos={ndos}, but HS has D={D}")
    if RHO.shape != (S, D, L) or OVB.shape != (S, D, L) or AGE.shape != (S, D, L):
        raise ValueError("Shape mismatch for 3D vars")
    if DIA.shape != (S, D, 5):
        raise ValueError(f"DIA must be (S,D,5); got {DIA.shape}")
    if SWE_mm.shape != (S, D) or HS_obs_cm.shape != (S, D):
        raise ValueError("Shape mismatch for 2D vars")

    with Dataset(out_nc, "w") as nc:
        # dims
        nc.createDimension("season", S)
        nc.createDimension("dos", D)
        nc.createDimension("layer", L)
        nc.createDimension("diag", 5)

        # coord vars
        v_season = nc.createVariable("season", "i4", ("season",))
        v_season[:] = np.asarray(seasons, dtype=np.int32)

        v_dos = nc.createVariable("dos", "i4", ("dos",))
        v_dos[:] = np.arange(1, D + 1, dtype=np.int32)
        v_dos.units = "day_of_season"

        v_layer = nc.createVariable("layer", "i4", ("layer",))
        v_layer[:] = np.arange(1, L + 1, dtype=np.int32)
        v_layer.units = "1"

        # global attrs to reconstruct dates from (season, dos)
        nc.setncattr("season_window_start_mmdd", "11-01")
        nc.setncattr("season_window_end_mmdd", "04-30")
        nc.setncattr("dos_definition",
                     "dos = 1 corresponds to Nov 1 of (season-1); dos increments daily until Apr 30 of (season).")

        # creators
        def v3(name: str, units: str):
            v = nc.createVariable(name, "f8", ("season", "dos", "layer"), fill_value=np.nan)
            v.units = units
            return v

        def v2(name: str, units: str):
            v = nc.createVariable(name, "f8", ("season", "dos"), fill_value=np.nan)
            v.units = units
            return v

        # variables
        vHS = v3("HS_layer_cm", "cm")
        vRHO = v3("RHO", "kg m-3")
        vOVB = v3("OVB", "1")
        vAGE = v3("AGE", "1")

        vDIA = nc.createVariable("DIA", "f8", ("season", "dos", "diag"), fill_value=np.nan)
        vDIA.units = "1"

        vSWE = v2("SWE_mm", "mm")
        vHSobs = v2("HS_obs_cm", "cm")

        # write
        vHS[:] = HS_cm
        vRHO[:] = RHO
        vOVB[:] = OVB
        vAGE[:] = AGE
        vDIA[:] = DIA
        vSWE[:] = SWE_mm
        vHSobs[:] = HS_obs_cm


# -----------------------------
# MAIN
# -----------------------------
def main() -> None:
    csv_files = sorted(INDIR.glob("*.csv"))
    if not csv_files:
        raise RuntimeError(f"No CSV files found in {INDIR}")

    for infile in csv_files:
        df = pd.read_csv(infile)

        date_col = "date" if "date" in df.columns else ("timestamp" if "timestamp" in df.columns else None)
        if date_col is None:
            raise ValueError(f"{infile.name}: no 'date' or 'timestamp' column")

        df["date_ts"] = pd.to_datetime(df[date_col])
        df["season"] = df["date_ts"].apply(season_label)
        station = station_name_from_file(infile)

        per_season: Dict[int, Dict[str, object]] = {}

        for season in sorted(df["season"].unique()):
            season = int(season)

            sdf = df[(df["season"] == season) & (df["date_ts"].apply(lambda d: in_season_window(d, season)))].copy()
            if len(sdf) < MIN_DAYS_PER_SEASON:
                continue
            sdf = sdf.sort_values("date_ts")

            hs_m = choose_hs_column(sdf)
            if np.all(~np.isfinite(hs_m)) or np.nansum(hs_m) == 0:
                continue

            hs_m = ensure_first_zero(hs_m)
            hs_cm_obs = np.where(np.isfinite(hs_m), hs_m * 100.0, np.nan)

            # run HS2SWE on the full season segment (time axis = observed days in sdf)
            idata = hs_cm_obs.reshape(-1, 1)

            try:
                SWE_2d, MR_list = HS2SWE(idata, return_MR=True)
            except Exception as e:
                print(f"Skipping {station} season {season}: HS2SWE error: {e}")
                continue

            SWE_mm = np.asarray(SWE_2d[:, 0], dtype=float)
            MR = MR_list[0]

            HS_layers_cm = np.asarray(MR["HS"], dtype=float)   # (L,Tn)
            RHO_layers = np.asarray(MR["RHO"], dtype=float)    # (L,Tn)
            OVB_layers = np.asarray(MR["OVB"], dtype=float)    # (L,Tn)
            AGE_layers = np.asarray(MR["AGE"], dtype=float)    # (L,Tn)
            DIA_layers = np.asarray(MR["DIA"], dtype=float)    # (5,Tn)

            # keep only seasons with enough "active" layers if desired
            nlay_active = max_active_layers(HS_layers_cm)
            if nlay_active < MIN_LAYERS_TO_SAVE:
                continue

            # DOS mapping for this season
            D_all = dos_length(season)
            dos_idx = dos_index(sdf["date_ts"], season)  # 1..D_all

            keep = np.where((dos_idx >= 1) & (dos_idx <= D_all))[0]
            if keep.size == 0:
                continue
            dos_idx = dos_idx[keep]
            # convert to 0-based for numpy indexing
            dos0 = dos_idx - 1

            # subset time dimension to keep and pack into DOS arrays
            hs_cm_obs = hs_cm_obs[keep]
            SWE_mm = SWE_mm[keep]
            HS_layers_cm = HS_layers_cm[:, keep]
            RHO_layers = RHO_layers[:, keep]
            OVB_layers = OVB_layers[:, keep]
            AGE_layers = AGE_layers[:, keep]
            DIA_layers = DIA_layers[:, keep]

            per_season[season] = {
                "dos_len": int(D_all),
                "dos0": dos0.astype(int),
                "HS_obs_cm": hs_cm_obs.astype(float),     # (Tkeep)
                "SWE_mm": SWE_mm.astype(float),           # (Tkeep)
                "HS_layers_cm": HS_layers_cm.astype(float),  # (L,Tkeep)
                "RHO": RHO_layers.astype(float),          # (L,Tkeep)
                "OVB": OVB_layers.astype(float),          # (L,Tkeep)
                "AGE": AGE_layers.astype(float),          # (L,Tkeep)
                "DIA": DIA_layers.astype(float),          # (5,Tkeep)
                "nlayer": int(HS_layers_cm.shape[0]),
            }
            print(f"OK: {station} season {season} (layers={HS_layers_cm.shape[0]}, active_layers={nlay_active})")

        if not per_season:
            print(f"No valid seasons for station: {station}")
            continue

        seasons_kept = sorted(per_season.keys())
        S = len(seasons_kept)

        # DOS length can differ across seasons (leap years include Feb 29),
        # so use the maximum DOS length across kept seasons.
        D = max(int(per_season[s]["dos_len"]) for s in seasons_kept)
        # max layers across seasons
        L = max(int(per_season[s]["nlayer"]) for s in seasons_kept)

        # allocate station arrays (season, dos, layer) and (season, dos, diag)
        HS_3d  = np.full((S, D, L), np.nan, dtype=float)
        RHO_3d = np.full((S, D, L), np.nan, dtype=float)
        OVB_3d = np.full((S, D, L), np.nan, dtype=float)
        AGE_3d = np.full((S, D, L), np.nan, dtype=float)
        DIA_3d = np.full((S, D, 5), np.nan, dtype=float)

        SWE_2d = np.full((S, D), np.nan, dtype=float)
        HSobs_2d = np.full((S, D), np.nan, dtype=float)
        # fill season by season (place each observed day into its DOS slot)
        for sidx, season in enumerate(seasons_kept):
            obj = per_season[season]
            dos0 = np.asarray(obj["dos0"], dtype=int)          # (Tkeep,)
            Ls = int(obj["nlayer"])

            # Safety guard: keep only indices that fit the station DOS axis.
            valid = (dos0 >= 0) & (dos0 < D)
            if not np.all(valid):
                print(f"Warning: {station} season {season} had out-of-range DOS indices; dropping them.")
                dos0 = dos0[valid]
                if dos0.size == 0:
                    continue

            # layers are (L,Tkeep) -> transpose to (Tkeep,L) for easy assignment
            HS_t  = np.asarray(obj["HS_layers_cm"], dtype=float).T   # (Tkeep,Ls)
            RHO_t = np.asarray(obj["RHO"], dtype=float).T
            OVB_t = np.asarray(obj["OVB"], dtype=float).T
            AGE_t = np.asarray(obj["AGE"], dtype=float).T

            if not np.all(valid):
                HS_t = HS_t[valid, :]
                RHO_t = RHO_t[valid, :]
                OVB_t = OVB_t[valid, :]
                AGE_t = AGE_t[valid, :]

            HS_3d[sidx, dos0, :Ls]  = HS_t[:, :Ls]
            RHO_3d[sidx, dos0, :Ls] = RHO_t[:, :Ls]
            OVB_3d[sidx, dos0, :Ls] = OVB_t[:, :Ls]
            AGE_3d[sidx, dos0, :Ls] = AGE_t[:, :Ls]

            # DIA is (5,Tkeep) -> (Tkeep,5)
            DIA_t = np.asarray(obj["DIA"], dtype=float).T
            if not np.all(valid):
                DIA_t = DIA_t[valid, :]
            DIA_3d[sidx, dos0, :] = DIA_t[:, :]

            SWE_vals = np.asarray(obj["SWE_mm"], dtype=float)
            HSobs_vals = np.asarray(obj["HS_obs_cm"], dtype=float)
            if not np.all(valid):
                SWE_vals = SWE_vals[valid]
                HSobs_vals = HSobs_vals[valid]

            SWE_2d[sidx, dos0] = SWE_vals
            HSobs_2d[sidx, dos0] = HSobs_vals
            HSobs_2d[sidx, dos0] = np.asarray(obj["HS_obs_cm"], dtype=float)

        out_nc = OUTDIR / f"{station}_hs2swe_allseasons.nc"
        write_station_hs2swe_nc(
            out_nc=out_nc,
            seasons=seasons_kept,
            ndos=D,
            HS_cm=HS_3d,
            RHO=RHO_3d,
            OVB=OVB_3d,
            AGE=AGE_3d,
            DIA=DIA_3d,
            SWE_mm=SWE_2d,
            HS_obs_cm=HSobs_2d,
        )
        print(f"Wrote station NetCDF: {out_nc}")

    print("Done.")


if __name__ == "__main__":
    main()

Running station series: 1
Elapsed time for series 1: 0.04896903038024902 s
Average time per series: 0.04907798767089844 s
OK: Adelboden season 2017 (layers=21, active_layers=20)
Running station series: 1
Elapsed time for series 1: 0.09296703338623047 s
Average time per series: 0.09300589561462402 s
OK: Adelboden season 2018 (layers=32, active_layers=31)
Running station series: 1
Elapsed time for series 1: 0.07196307182312012 s
Average time per series: 0.07206606864929199 s
OK: Adelboden season 2019 (layers=23, active_layers=22)
Running station series: 1
Elapsed time for series 1: 0.04072880744934082 s
Average time per series: 0.04075980186462402 s
OK: Adelboden season 2020 (layers=5, active_layers=4)
Running station series: 1
Elapsed time for series 1: 0.06671881675720215 s
Average time per series: 0.06674408912658691 s
OK: Adelboden season 2021 (layers=24, active_layers=23)
Running station series: 1
Elapsed time for series 1: 0.08025097846984863 s
Average time per series: 0.0803067684